# Notebook 11: Infinity-2B GGUF prompt-stability study

This notebook tests whether an Infinity-2B GGUF content failure is caused by an underspecified prompt or by the model itself. It generates the 48 object/scene cases from Notebook 9 twice: once with the original short prompt and once with a more specific, style-free prompt.

The comparison is controlled: every pair uses the same seed, resolution preset, CFG, sampling temperature, top-k, and top-p. There is no style image, feature injection, or metric in this notebook.


In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import time

# All files are kept under one directory so the notebook can be rerun safely.
ROOT = Path('/content/notebook_11_infinity2b_gguf_prompt_stability')
PORT_DIR = ROOT / 'gguf_port'
OFFICIAL_DIR = PORT_DIR / 'Infinity'
ASSET_DIR = ROOT / 'assets'
OUTPUT_DIR = ROOT / 'outputs'
for path in (PORT_DIR, ASSET_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'
MODEL_PN = '0.25M'  # Colab-friendly 512px adaptation; the paper reports 1M/1024px.
CFG_SCALE = 1.0
TAU = 0.1
SEED = 42
T5_DEVICE = 'cuda'  # T4/Colab: keep T5 off host RAM; use 'cpu' only with a high-RAM runtime.
print('ROOT:', ROOT)
print('MODEL_PN:', MODEL_PN, '| CFG:', CFG_SCALE, '| TAU:', TAU, '| SEED:', SEED, '| T5:', T5_DEVICE)

In [ ]:
# Check the runtime before installing anything.
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 2**30, 2))
else:
    print('WARNING: no GPU detected; inference will be extremely slow.')

In [ ]:
# Install the packages used by the GGUF loader.
# We intentionally do not install torch or flash-attn here: Colab already ships torch,
# and the GGUF loader falls back to PyTorch SDPA when flash-attn is unavailable.
packages = [
    'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia',
    'gputil', 'colorama', 'omegaconf', 'timm==0.9.6',
    'decord', 'pytz', 'imageio', 'einops', 'opencv-python', 'accelerate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependency installation finished.')

In [ ]:
# Clone the official Python architecture only once.
if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

# Download only the files needed from the unofficial GGUF repository.
from huggingface_hub import hf_hub_download

def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(
        repo_id=GGUF_REPO,
        filename=filename,
        local_dir=str(target_dir),
    ))

PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
PATCH_DIR = ROOT / 'gguf_patched_source'
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', PATCH_DIR)
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', PATCH_DIR)

# The GGUF repository includes patched source files with optional attention fallbacks.
# Copy them over the matching files in the official source tree.
official_basic = OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py'
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_BASIC, official_basic)
shutil.copy2(PATCHED_INFINITY, official_infinity)

# The patched attention module may intentionally expose flash_attn_func=None.
# Guard the official constructor so it selects the PyTorch SDPA fallback safely.
infinity_source = official_infinity.read_text()
old_attention_guard = "customized_kernel_installed = any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
new_attention_guard = "customized_kernel_installed = flash_attn_func is not None and any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
if old_attention_guard not in infinity_source:
    raise RuntimeError('Expected optional-attention guard was not found in patched infinity.py')
official_infinity.write_text(infinity_source.replace(old_attention_guard, new_attention_guard, 1))
INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

print('GGUF model:', INFINITY_GGUF)
print('T5 encoder:', T5_GGUF)
print('VAE:', VAE_PATH)
print('Loader:', PORT_SCRIPT)
print('Loader utility:', PORT_UTILS)

In [ ]:
# Verify the expected files before importing the custom loader.
required_files = [PORT_SCRIPT, PORT_UTILS, PATCHED_BASIC, PATCHED_INFINITY, INFINITY_GGUF, T5_GGUF, VAE_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

for path in required_files:
    print(f'{path.name:40s} {path.stat().st_size / 2**30:.3f} GiB')

assert (OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py').exists(), 'Official Infinity source is incomplete.'
print('All GGUF, VAE, and official source files are present.')

## Memory-efficient T5 loading

Flan-T5-XL is kept because Infinity-2B expects its learned 2048-dimensional text features. The loader streams GGUF tensors directly into the encoder to reduce Colab host-RAM peak usage.


In [ ]:
import math
import numpy as np
import gguf

def load_t5_encoder_streaming(gguf_path, device='cpu'):
    from gguf import GGUFReader
    from transformers import T5Config, T5EncoderModel

    key_map = {
        'enc.': 'encoder.',
        '.blk.': '.block.',
        'token_embd': 'shared',
        'output_norm': 'final_layer_norm',
        'attn_q': 'layer.0.SelfAttention.q',
        'attn_k': 'layer.0.SelfAttention.k',
        'attn_v': 'layer.0.SelfAttention.v',
        'attn_o': 'layer.0.SelfAttention.o',
        'attn_norm': 'layer.0.layer_norm',
        'attn_rel_b': 'layer.0.SelfAttention.relative_attention_bias',
        'ffn_up': 'layer.1.DenseReluDense.wi_1',
        'ffn_down': 'layer.1.DenseReluDense.wo',
        'ffn_gate': 'layer.1.DenseReluDense.wi_0',
        'ffn_norm': 'layer.1.layer_norm',
    }

    config = T5Config.from_pretrained('google/flan-t5-xl')
    try:
        from accelerate import init_empty_weights
        with init_empty_weights():
            model = T5EncoderModel(config)
        # Materialize directly as FP16 to avoid allocating a full FP32 T5.
        model = model.to(dtype=torch.float16)
        model.to_empty(device=device)
    except Exception as exc:
        raise RuntimeError(
            'Streaming T5 loading requires the accelerate package and empty-weight support. '
            'Restart the runtime and rerun the dependency cell.'
        ) from exc

    model.eval()
    model.requires_grad_(False)
    parameter_refs = dict(model.named_parameters())
    buffer_refs = dict(model.named_buffers())
    reader = GGUFReader(str(gguf_path))
    quantized_types = {gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16}
    loaded = 0
    skipped = []

    print(f'[Streaming T5 load] {gguf_path} -> {device}')
    with torch.inference_mode():
        for tensor in reader.tensors:
            name = tensor.name
            for old_key, new_key in key_map.items():
                name = name.replace(old_key, new_key)
            shape = torch.Size(tuple(int(v) for v in reversed(tensor.shape)))
            raw = torch.from_numpy(np.array(tensor.data))
            is_quantized = tensor.tensor_type not in quantized_types
            if is_quantized:
                quant_param = gguf_loader.GGUFParameter(raw, quant_type=tensor.tensor_type)
                value = gguf_loader.dequantize_gguf_tensor(quant_param, target_dtype=torch.float16)
            else:
                value = raw.to(dtype=torch.float16)
            if value.numel() != math.prod(shape):
                skipped.append((name, 'numel mismatch'))
                del raw, value
                continue
            value = value.reshape(shape)
            target = parameter_refs.get(name)
            if target is None:
                target = buffer_refs.get(name)
            if target is None or tuple(target.shape) != tuple(shape):
                skipped.append((name, 'missing or shape mismatch'))
                del raw, value
                continue
            target.data.copy_(value.to(device=target.device, dtype=target.dtype))
            loaded += 1
            del raw, value

    del reader, parameter_refs, buffer_refs
    gc.collect()
    # The model was materialized on the requested device already.
    # Keep this safety path for unusual device-string inputs.
    if str(next(model.parameters()).device) != str(torch.device(device)):
        model.to(device)
    model.eval()
    model.requires_grad_(False)
    print(f'[Streaming T5 load complete] tensors loaded: {loaded}, skipped: {len(skipped)}')
    if skipped:
        print('First skipped tensors:', skipped[:5])
    return model

print('Memory-efficient T5 loader is ready.')

## Import the GGUF loader

The following compatibility patch preserves the same PyTorch SDPA fallback used in Notebook 9.


In [ ]:
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_source = PORT_SCRIPT.read_text()
compat_pattern = r"\n    # Apply NumPy 2\.0 compatibility patch.*?\n    # Load GGUF state dict"
loader_source, replacements = re.subn(
    compat_pattern,
    '\n    # NumPy compatibility is handled by the installed gguf package.\n    # Load GGUF state dict',
    loader_source,
    count=1,
    flags=re.S,
)
print('Removed obsolete NumPy compatibility block:', replacements == 1)

PATCHED_LOADER = PORT_DIR / 'generate_image_2b_q8_gguf_colab.py'
PATCHED_LOADER.write_text(loader_source)
spec = importlib.util.spec_from_file_location('infinity_gguf_colab_loader', PATCHED_LOADER)
gguf_loader = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gguf_loader
spec.loader.exec_module(gguf_loader)
print('Custom GGUF loader imported successfully.')

## Load Infinity-2B GGUF

Use a GPU Colab runtime. The 0.25M preset is selected to match the prior Notebook 9 experiments.


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('A CUDA GPU is required for practical inference. Select a GPU runtime and rerun.')

print('[1/4] Loading T5 tokenizer...')
text_tokenizer = gguf_loader.load_t5_tokenizer_from_gguf(str(T5_GGUF))

print(f'[2/4] Streaming quantized T5 encoder to {T5_DEVICE}...')
text_encoder = load_t5_encoder_streaming(str(T5_GGUF), device=T5_DEVICE)

print('[3/4] Loading VAE on GPU...')
vae = gguf_loader.load_vae(str(VAE_PATH), vae_type=32, device=DEVICE)

print('[4/4] Loading quantized Infinity-2B transformer on GPU...')
infinity_model = gguf_loader.load_infinity_from_gguf(
    str(INFINITY_GGUF),
    vae=vae,
    device=DEVICE,
    model_type='infinity_2b',
    text_channels=2048,
    pn=MODEL_PN,
)

infinity_model.eval()
vae.eval()
print('All components loaded successfully.')

In [ ]:
# Build the official dynamic-resolution schedule for the selected preset.
import numpy as np
from infinity.utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates

ASPECT_RATIO = 1.0
h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - ASPECT_RATIO))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template][MODEL_PN]['scales']
scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]
print('Aspect ratio:', h_div_w_template)
print('Preset:', MODEL_PN)
print('Scale schedule:', scale_schedule)

In [ ]:
from PIL import Image
import inspect
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Strict shared sampling configuration for every short/detailed pair.
CFG_SCALE = 1.0
TAU = 0.1               # Infinity sampling temperature, not an attention temperature.
TOP_K = 600
TOP_P = 0.95
SEED = 42
FORCE_REGENERATE = False

RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / 'Infinity_outputs' / 'notebook_11_prompt_stability_study'
SHORT_DIR = OUTPUT_ROOT / 'short_prompts'
DETAILED_DIR = OUTPUT_ROOT / 'detailed_prompts'
GALLERY_DIR = OUTPUT_ROOT / 'galleries'
for path in (SHORT_DIR, DETAILED_DIR, GALLERY_DIR):
    path.mkdir(parents=True, exist_ok=True)

# The GGUF helper may use model-level sampling defaults. Set them explicitly as
# a fallback, then pass them directly whenever the helper exposes these arguments.
infinity_model.top_k = TOP_K
infinity_model.top_p = TOP_P
print(f'Preset={MODEL_PN} | CFG={CFG_SCALE} | tau={TAU} | top-k={TOP_K} | top-p={TOP_P} | seed={SEED}')
print('Output root:', OUTPUT_ROOT)


def tensor_to_pil(image):
    if isinstance(image, (list, tuple)):
        image = image[0]
    tensor = image.detach().float().cpu() if torch.is_tensor(image) else torch.as_tensor(image).float()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Unexpected image shape: {tuple(tensor.shape)}')
    if tensor.shape[0] in (1, 3, 4):
        tensor = tensor.permute(1, 2, 0)
    if tensor.shape[-1] == 1:
        tensor = tensor.repeat(1, 1, 3)
    if tensor.shape[-1] > 3:
        tensor = tensor[..., :3]
    low, high = float(tensor.min()), float(tensor.max())
    if low < -0.05:
        tensor = (tensor + 1.0) / 2.0
    elif high > 1.05:
        tensor = tensor / 255.0
    return Image.fromarray((tensor.clamp(0, 1).numpy() * 255).round().astype('uint8'), mode='RGB')


def generate_one(prompt, seed=SEED):
    """Generate one content image under the fixed prompt-stability settings."""
    kwargs = {
        'cfg_scale': CFG_SCALE,
        'tau': TAU,
        'seed': seed,
        'scale_schedule': scale_schedule,
        'vae_type': 32,
        'device': DEVICE,
    }
    signature = inspect.signature(gguf_loader.generate_image)
    optional_sampling = {'top_k': TOP_K, 'top_p': TOP_P}
    accepts_var_kwargs = any(param.kind == inspect.Parameter.VAR_KEYWORD for param in signature.parameters.values())
    for name, value in optional_sampling.items():
        if name in signature.parameters or accepts_var_kwargs:
            kwargs[name] = value
    with torch.inference_mode():
        image = gguf_loader.generate_image(
            infinity_model, vae, text_tokenizer, text_encoder, prompt, **kwargs,
        )
    result = tensor_to_pil(image)
    del image
    gc.collect()
    torch.cuda.empty_cache()
    return result


def load_or_generate(prompt, path):
    if path.exists() and not FORCE_REGENERATE:
        with Image.open(path) as image:
            return image.convert('RGB').copy()
    image = generate_one(prompt)
    image.save(path)
    return image


## Prompt pairs

The short prompts are copied from Notebook 9. Detailed prompts add ordinary object attributes, setting, or composition only; they deliberately do not name an artistic style.

Interpretation: if the detailed prompt consistently fixes a case, the short prompt was likely underspecified for this model. If both versions fail in the same way, treat it as a model/vocabulary/quantization or sampling limitation rather than a style-transfer issue.


In [ ]:
PROMPT_PAIRS = [
    {'case_id': 's01_p01', 'session': 'colorful origami', 'short_prompt': 'a kettle', 'detailed_prompt': 'a stainless steel kettle on a wooden kitchen table'},
    {'case_id': 's01_p02', 'session': 'colorful origami', 'short_prompt': 'an armchair', 'detailed_prompt': 'a beige armchair in a bright living room'},
    {'case_id': 's01_p03', 'session': 'colorful origami', 'short_prompt': 'a table', 'detailed_prompt': 'a wooden dining table in a sunlit room'},
    {'case_id': 's01_p04', 'session': 'colorful origami', 'short_prompt': 'a pair of shoes', 'detailed_prompt': 'a pair of white sneakers on a clean floor'},
    {'case_id': 's01_p05', 'session': 'colorful origami', 'short_prompt': 'a car', 'detailed_prompt': 'a red car parked on a city street'},
    {'case_id': 's01_p06', 'session': 'colorful origami', 'short_prompt': 'a ball', 'detailed_prompt': 'a colorful ball on green grass'},

    {'case_id': 's02_p01', 'session': 'flat cartoon vector', 'short_prompt': 'a pen', 'detailed_prompt': 'a blue pen on a white desk'},
    {'case_id': 's02_p02', 'session': 'flat cartoon vector', 'short_prompt': 'a coral', 'detailed_prompt': 'a coral reef underwater'},
    {'case_id': 's02_p03', 'session': 'flat cartoon vector', 'short_prompt': 'a bell', 'detailed_prompt': 'a brass bell on a wooden table'},
    {'case_id': 's02_p04', 'session': 'flat cartoon vector', 'short_prompt': 'a helmet', 'detailed_prompt': 'a yellow motorcycle helmet on a table'},
    {'case_id': 's02_p05', 'session': 'flat cartoon vector', 'short_prompt': 'a bird', 'detailed_prompt': 'a small bird perched on a branch'},
    {'case_id': 's02_p06', 'session': 'flat cartoon vector', 'short_prompt': 'a book', 'detailed_prompt': 'an open book on a wooden desk'},

    {'case_id': 's03_p01', 'session': 'northern renaissance', 'short_prompt': 'an hourglass', 'detailed_prompt': 'a glass hourglass with sand on a wooden table'},
    {'case_id': 's03_p02', 'session': 'northern renaissance', 'short_prompt': 'a telescope', 'detailed_prompt': 'a brass telescope on a tripod under a night sky'},
    {'case_id': 's03_p03', 'session': 'northern renaissance', 'short_prompt': 'a castle', 'detailed_prompt': 'a stone castle on a hill beneath a blue sky'},
    {'case_id': 's03_p04', 'session': 'northern renaissance', 'short_prompt': 'a piano', 'detailed_prompt': 'a black grand piano in a music room'},
    {'case_id': 's03_p05', 'session': 'northern renaissance', 'short_prompt': 'a horse', 'detailed_prompt': 'a brown horse standing in a grassy field'},
    {'case_id': 's03_p06', 'session': 'northern renaissance', 'short_prompt': 'a train', 'detailed_prompt': 'a steam train traveling through green countryside'},

    {'case_id': 's04_p01', 'session': 'intricate line art', 'short_prompt': 'a pine tree', 'detailed_prompt': 'a tall pine tree in a forest'},
    {'case_id': 's04_p02', 'session': 'intricate line art', 'short_prompt': 'a bear', 'detailed_prompt': 'a brown bear standing in a forest'},
    {'case_id': 's04_p03', 'session': 'intricate line art', 'short_prompt': 'a pile of pebbles', 'detailed_prompt': 'a pile of smooth pebbles on a sandy beach'},
    {'case_id': 's04_p04', 'session': 'intricate line art', 'short_prompt': 'a mountain range', 'detailed_prompt': 'a mountain range beneath a blue sky'},
    {'case_id': 's04_p05', 'session': 'intricate line art', 'short_prompt': 'a pair of boots', 'detailed_prompt': 'a pair of brown leather boots on a wooden floor'},

    {'case_id': 's05_p01', 'session': 'geometric flat', 'short_prompt': 'an oval mirror', 'detailed_prompt': 'an oval mirror hanging on a white wall'},
    {'case_id': 's05_p02', 'session': 'geometric flat', 'short_prompt': 'a leaf', 'detailed_prompt': 'a green leaf on a white background'},
    {'case_id': 's05_p03', 'session': 'geometric flat', 'short_prompt': 'a lemon', 'detailed_prompt': 'a yellow lemon on a wooden table'},
    {'case_id': 's05_p04', 'session': 'geometric flat', 'short_prompt': 'a moose', 'detailed_prompt': 'a moose standing in a grassy meadow'},
    {'case_id': 's05_p05', 'session': 'geometric flat', 'short_prompt': 'a motorbike', 'detailed_prompt': 'a red motorbike parked on a street'},

    {'case_id': 's06_p01', 'session': 'metallic 3D', 'short_prompt': 'a deer', 'detailed_prompt': 'a deer standing in a forest clearing'},
    {'case_id': 's06_p02', 'session': 'metallic 3D', 'short_prompt': 'a crab', 'detailed_prompt': 'a red crab on a sandy beach'},
    {'case_id': 's06_p03', 'session': 'metallic 3D', 'short_prompt': 'a mountain peak', 'detailed_prompt': 'a snow-covered mountain peak under a blue sky'},
    {'case_id': 's06_p04', 'session': 'metallic 3D', 'short_prompt': 'a fish', 'detailed_prompt': 'a silver fish swimming underwater'},
    {'case_id': 's06_p05', 'session': 'metallic 3D', 'short_prompt': 'a drone', 'detailed_prompt': 'a quadcopter drone flying over a field'},

    {'case_id': 's07_p01', 'session': 'neon splash comic', 'short_prompt': 'a boom box', 'detailed_prompt': 'a retro boom box on a wooden table'},
    {'case_id': 's07_p02', 'session': 'neon splash comic', 'short_prompt': 'a cube robot', 'detailed_prompt': 'a small cube-shaped robot on a desk'},
    {'case_id': 's07_p03', 'session': 'neon splash comic', 'short_prompt': 'a spaceship', 'detailed_prompt': 'a spaceship flying above a planet'},
    {'case_id': 's07_p04', 'session': 'neon splash comic', 'short_prompt': 'a snowboard', 'detailed_prompt': 'a blue snowboard standing upright in snow'},
    {'case_id': 's07_p05', 'session': 'neon splash comic', 'short_prompt': 'a jeep', 'detailed_prompt': 'a rugged jeep driving on a dirt road'},

    {'case_id': 's08_p01', 'session': 'architectural line art', 'short_prompt': 'a chef hat', 'detailed_prompt': 'a white chef hat on a kitchen counter'},
    {'case_id': 's08_p02', 'session': 'architectural line art', 'short_prompt': 'a phoenix', 'detailed_prompt': 'a phoenix flying above mountains at sunset'},
    {'case_id': 's08_p03', 'session': 'architectural line art', 'short_prompt': 'a teddy bear', 'detailed_prompt': 'a brown teddy bear sitting on a bed'},
    {'case_id': 's08_p04', 'session': 'architectural line art', 'short_prompt': 'a turtle', 'detailed_prompt': 'a sea turtle swimming underwater'},
    {'case_id': 's08_p05', 'session': 'architectural line art', 'short_prompt': 'an armchair', 'detailed_prompt': 'a beige armchair in a bright living room'},

    {'case_id': 's09_p01', 'session': 'retro sci-fi', 'short_prompt': 'an observatory', 'detailed_prompt': 'an observatory on a hill under a starry night sky'},
    {'case_id': 's09_p02', 'session': 'retro sci-fi', 'short_prompt': 'a lighthouse', 'detailed_prompt': 'a lighthouse beside the sea on a rocky coast'},
    {'case_id': 's09_p03', 'session': 'retro sci-fi', 'short_prompt': 'a taxi', 'detailed_prompt': 'a yellow taxi driving on a city street'},
    {'case_id': 's09_p04', 'session': 'retro sci-fi', 'short_prompt': 'a dragon', 'detailed_prompt': 'a dragon standing on a rocky mountain'},
    {'case_id': 's09_p05', 'session': 'retro sci-fi', 'short_prompt': 'a domed city', 'detailed_prompt': 'a futuristic domed city beneath a clear sky'},
]

assert len(PROMPT_PAIRS) == 48
assert len({pair['case_id'] for pair in PROMPT_PAIRS}) == 48
print(f'Prepared {len(PROMPT_PAIRS)} controlled prompt pairs.')


## Generate short and detailed prompts

This produces 96 images in total. Each image is written to disk immediately. If Colab disconnects, rerun this cell: saved images are loaded from disk and only missing images are generated.


In [ ]:
PAIR_RESULTS = {}

for pair in tqdm(PROMPT_PAIRS, desc='Generating prompt-stability pairs'):
    short_path = SHORT_DIR / f"{pair['case_id']}.png"
    detailed_path = DETAILED_DIR / f"{pair['case_id']}.png"
    PAIR_RESULTS[pair['case_id']] = {
        'short': load_or_generate(pair['short_prompt'], short_path),
        'detailed': load_or_generate(pair['detailed_prompt'], detailed_path),
    }

print(f'Prompt pairs available: {len(PAIR_RESULTS)} / {len(PROMPT_PAIRS)}')


## Side-by-side review galleries

Read each row as one controlled pair. A clear improvement only in the detailed column supports the hypothesis that the original object prompt was underspecified. Similar failure in both columns points to an Infinity-2B GGUF/content-model limitation for that concept under this configuration.


In [ ]:
def _plot_image(axis, image):
    axis.imshow(image)
    axis.set_xticks([])
    axis.set_yticks([])


def show_session_gallery(session_id):
    session_pairs = [pair for pair in PROMPT_PAIRS if pair['case_id'].startswith(f's{session_id:02d}_')]
    session_name = session_pairs[0]['session']
    figure, axes = plt.subplots(len(session_pairs), 3, figsize=(12, 4.2 * len(session_pairs)), squeeze=False)
    figure.suptitle(
        f'Infinity-2B GGUF prompt-stability comparison | {session_name} | '
        f'CFG {CFG_SCALE}, tau {TAU}, top-k {TOP_K}, top-p {TOP_P}, seed {SEED}',
        fontsize=14,
        y=0.995,
    )
    for row, pair in enumerate(session_pairs):
        pair_images = PAIR_RESULTS[pair['case_id']]
        axes[row, 0].axis('off')
        axes[row, 0].text(
            0.5, 0.58, 'SHORT PROMPT', ha='center', va='center', fontsize=10, weight='bold',
            transform=axes[row, 0].transAxes,
        )
        axes[row, 0].text(
            0.5, 0.43, f'"{pair["short_prompt"]}"', ha='center', va='center', fontsize=13,
            wrap=True, transform=axes[row, 0].transAxes,
        )
        axes[row, 0].text(
            0.5, 0.18, 'DETAILED PROMPT', ha='center', va='center', fontsize=10, weight='bold',
            transform=axes[row, 0].transAxes,
        )
        axes[row, 0].text(
            0.5, 0.06, f'"{pair["detailed_prompt"]}"', ha='center', va='center', fontsize=10,
            wrap=True, transform=axes[row, 0].transAxes,
        )
        _plot_image(axes[row, 1], pair_images['short'])
        _plot_image(axes[row, 2], pair_images['detailed'])
        if row == 0:
            axes[row, 1].set_title('Short-prompt image', fontsize=11)
            axes[row, 2].set_title('Detailed-prompt image', fontsize=11)
    figure.tight_layout()
    save_path = GALLERY_DIR / f'session_{session_id:02d}_short_vs_detailed.png'
    figure.savefig(save_path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    print('Saved:', save_path)


for session_id in range(1, 10):
    show_session_gallery(session_id)


## Download outputs

This creates one ZIP containing all 96 individual images and the session galleries.


In [ ]:
from google.colab import files

archive_base = Path('/content/notebook_11_prompt_stability_outputs')
archive_file = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT)
print('Created:', archive_file)
files.download(archive_file)
